# Предобработка — merged.csv

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/processed/merged.csv', parse_dates=['date'])
print('Исходный датасет:', df.shape)
print('Пропуски:')
print(df.isnull().sum())

Исходный датасет: (16832702, 12)
Пропуски:
date            0
price           0
level           0
levels          0
rooms           0
area            0
kitchen_area    0
geo_lat         0
geo_lon         0
object_type     0
region          0
source          0
dtype: int64


## 1. Выбросы по цене

In [3]:
print(f'price = 0:        {(df["price"] == 0).sum():,}')
print(f'price < 100k:     {(df["price"] < 100_000).sum():,}')
print(f'price > 1 млрд:   {(df["price"] > 1_000_000_000).sum():,}')

mask_price = (df['price'] >= 100_000) & (df['price'] <= 1_000_000_000)
print(f'\nУдаляем: {(~mask_price).sum():,} записей')
df = df[mask_price].copy().reset_index(drop=True)
print(f'После: {len(df):,}')

price = 0:        8,123
price < 100k:     14,999
price > 1 млрд:   1,525

Удаляем: 16,524 записей
После: 16,816,178


## 2. kitchen_area отрицательная → NaN

In [4]:
n_neg = (df['kitchen_area'] < 0).sum()
print(f'Отрицательных kitchen_area: {n_neg:,}')
df.loc[df['kitchen_area'] < 0, 'kitchen_area'] = np.nan
print(f'Заменено на NaN: {df["kitchen_area"].isna().sum():,}')

Отрицательных kitchen_area: 1,084,561
Заменено на NaN: 1,084,561


## 3. rooms: -2 удалить, -1 → 0 (студия)

In [5]:
print(f'rooms = -2: {(df["rooms"] == -2).sum():,} → удаляем')
print(f'rooms = -1: {(df["rooms"] == -1).sum():,} → перекодируем в 0 (студия)')

df = df[df['rooms'] != -2].copy().reset_index(drop=True)
df.loc[df['rooms'] == -1, 'rooms'] = 0

print(f'\nrooms после очистки: {sorted(df["rooms"].unique())}')
print(f'Записей: {len(df):,}')

rooms = -2: 343 → удаляем
rooms = -1: 1,144,225 → перекодируем в 0 (студия)

rooms после очистки: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Записей: 16,815,835


## 4. Добавление признака dist_to_center

In [6]:
# Координаты административных центров регионов
region_centers = {
    1: (44.609764, 40.100516),   # Майкоп
    2: (54.735152, 55.958736),   # Уфа
    3: (51.834417, 107.584219),  # Улан-Удэ
    4: (51.957169, 85.960649),   # Барнаул
    5: (42.980139, 47.502079),   # Махачкала
    6: (43.168722, 44.820922),   # Назрань
    7: (43.484983, 43.606726),   # Нальчик
    8: (46.308352, 44.256962),   # Элиста
    9: (43.888578, 42.019895),   # Черкесск
    10: (61.784916, 34.346218),  # Петрозаводск
    11: (61.668804, 50.836499),  # Сыктывкар
    12: (56.634483, 47.898661),  # Йошкар-Ола
    13: (54.184271, 45.183786),  # Саранск
    14: (62.028103, 129.732663), # Якутск
    15: (43.024122, 44.682006),  # Владикавказ
    16: (55.796127, 49.106414),  # Казань
    17: (51.719543, 94.437581),  # Кызыл
    18: (56.852738, 53.204595),  # Ижевск
    19: (53.721152, 91.442396),  # Абакан
    20: (43.317992, 45.698197),  # Грозный
    21: (56.139918, 47.247699),  # Чебоксары
    22: (53.346785, 83.776856),  # Барнаул (Алтайский край)
    23: (45.035470, 38.975313),  # Краснодар
    24: (56.010563, 92.852572),  # Красноярск
    25: (43.115536, 131.885485), # Владивосток
    26: (45.044490, 41.969080),  # Ставрополь
    27: (48.480268, 135.071917), # Хабаровск
    28: (50.290658, 127.527173), # Благовещенск
    29: (64.539393, 40.518735),  # Архангельск
    30: (46.347869, 48.033574),  # Астрахань
    31: (50.597467, 36.588044),  # Белгород
    32: (53.243521, 34.363997),  # Брянск
    33: (56.129057, 40.406635),  # Владимир
    34: (48.707103, 44.516939),  # Волгоград
    35: (59.220694, 39.891523),  # Вологда
    36: (51.671994, 39.184288),  # Воронеж
    37: (57.000348, 40.973490),  # Иваново
    38: (52.286387, 104.280654), # Иркутск
    39: (54.710162, 20.510137),  # Калининград
    40: (54.513845, 36.261360),  # Калуга
    41: (53.024284, 158.643743), # Петропавловск-Камчатский
    42: (55.354968, 86.086847),  # Кемерово
    43: (58.603600, 49.668054),  # Киров
    44: (57.767918, 40.926985),  # Кострома
    45: (55.441002, 65.341139),  # Курган
    46: (51.730361, 36.192647),  # Курск
    47: (59.919543, 30.423396),  # Гатчина (Ленобласть)
    48: (52.608826, 39.599455),  # Липецк
    49: (59.568469, 150.793557), # Магадан
    50: (55.811960, 37.415010),  # Красногорск (Московская обл)
    51: (68.969862, 33.074918),  # Мурманск
    52: (56.326797, 44.007157),  # Нижний Новгород
    53: (58.521475, 31.269057),  # Великий Новгород
    54: (54.989347, 82.904632),  # Новосибирск
    55: (54.989342, 73.368212),  # Омск
    56: (51.768199, 55.096957),  # Оренбург
    57: (52.967631, 36.069622),  # Орёл
    58: (53.194879, 45.019571),  # Пенза
    59: (58.010455, 56.229443),  # Пермь
    60: (57.819365, 28.331786),  # Псков
    61: (47.222078, 39.718803),  # Ростов-на-Дону
    62: (54.629216, 39.741480),  # Рязань
    63: (53.195878, 50.100202),  # Самара
    64: (51.533103, 46.034158),  # Саратов
    65: (46.959118, 142.738023), # Южно-Сахалинск
    66: (56.838011, 60.597474),  # Екатеринбург
    67: (54.782635, 32.045251),  # Смоленск
    68: (52.721219, 41.452274),  # Тамбов
    69: (56.859611, 35.911896),  # Тверь
    70: (56.484645, 84.947649),  # Томск
    71: (54.193122, 37.617348),  # Тула
    72: (57.153033, 68.974689),  # Тюмень
    73: (54.317002, 48.402243),  # Ульяновск
    74: (55.159902, 61.402554),  # Челябинск
    75: (52.033635, 113.499918), # Чита
    76: (57.626569, 39.893787),  # Ярославль
    77: (55.755826, 37.617300),  # Москва
    78: (59.939095, 30.315868),  # Санкт-Петербург
    79: (48.480266, 132.938833), # Биробиджан
    83: (67.638011, 53.006717),  # Нарьян-Мар
    86: (61.004147, 69.018978),  # Ханты-Мансийск
    87: (64.733877, 177.514365), # Анадырь
    89: (66.529818, 66.613497),  # Салехард
    91: (45.186360, 33.379296),  # Симферополь
    92: (44.616648, 33.525369),  # Севастополь
}

In [7]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

center_lat = df['region'].map(lambda r: region_centers.get(r, (None, None))[0])
center_lon = df['region'].map(lambda r: region_centers.get(r, (None, None))[1])

df['dist_to_center'] = haversine(
    df['geo_lat'].values, df['geo_lon'].values,
    center_lat.values.astype(float), center_lon.values.astype(float)
)

print('dist_to_center (км):')
print(df['dist_to_center'].describe().round(1))
print(f'NaN: {df["dist_to_center"].isna().sum()}')

dist_to_center (км):
count    16815835.0
mean           44.0
std            89.5
min             0.0
25%             4.6
50%             8.6
75%            34.4
max          7286.3
Name: dist_to_center, dtype: float64
NaN: 0


## 5. Итоговая проверка

In [8]:
print('Итоговый датасет:', df.shape)
print()
print('Пропуски:')
print(df.isnull().sum())
print()
print('Типы данных:')
print(df.dtypes)
print()
print('Статистика:')
display(df[['price','area','kitchen_area','rooms','level','levels','dist_to_center']].describe().round(2))

Итоговый датасет: (16815835, 13)

Пропуски:
date                    0
price                   0
level                   0
levels                  0
rooms                   0
area                    0
kitchen_area      1084561
geo_lat                 0
geo_lon                 0
object_type             0
region                  0
source                  0
dist_to_center          0
dtype: int64

Типы данных:
date              datetime64[us]
price                      int64
level                      int64
levels                     int64
rooms                      int64
area                     float64
kitchen_area             float64
geo_lat                  float64
geo_lon                  float64
object_type                int64
region                   float64
source                       str
dist_to_center           float64
dtype: object

Статистика:


,price,area,kitchen_area,rooms,level,levels,dist_to_center
count,1.681584e+07,16815835.00,15731274.00,16815835.00,16815835.00,16815835.00,16815835.00
mean,5.739053e+06,53.36,8.70,1.79,6.36,11.65,44.00
std,1.205330e+07,29.20,7.96,0.99,5.18,7.01,89.53
min,1.000000e+05,0.07,0.00,0.00,0.00,0.00,0.00
25%,2.350000e+06,37.00,5.40,1.00,2.00,5.00,4.63
50%,3.650000e+06,47.00,8.50,2.00,5.00,10.00,8.61
75%,5.978700e+06,63.00,12.00,2.00,9.00,17.00,34.36
max,1.000000e+09,7856.00,9999.00,10.00,50.00,50.00,7286.26


## 6. Сохранение

In [9]:
output_path = '../data/processed/merged_clean.csv'
df.to_csv(output_path, index=False)
print(f'Сохранено: {output_path}')
print(f'Итог: {len(df):,} строк, {df.shape[1]} колонок')
print(f'Колонки: {list(df.columns)}')

Сохранено: ../data/processed/merged_clean.csv
Итог: 16,815,835 строк, 13 колонок
Колонки: ['date', 'price', 'level', 'levels', 'rooms', 'area', 'kitchen_area', 'geo_lat', 'geo_lon', 'object_type', 'region', 'source', 'dist_to_center']
